# Statistical Business Analysis — Week 7

## Project Overview
This project applies introductory statistical methods to business sales data. It covers descriptive statistics, distribution analysis, correlation, hypothesis testing, confidence intervals, and linear regression.

### Objectives
- Calculate descriptive statistics.
- Examine the distribution of sales.
- Measure relationships among business metrics.
- Perform at least three hypothesis tests.
- Calculate a 95% confidence interval and margin of error.
- Build a linear regression model and evaluate R-squared.
- Translate statistical results into practical business insights.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_csv("business_data.csv")
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 1. Descriptive Statistics

In [ ]:
numeric_cols = ["Sales", "Marketing_Spend", "Customers"]

descriptive = df[numeric_cols].describe().T
descriptive["mode"] = [df[c].mode().iloc[0] for c in numeric_cols]
descriptive["variance"] = [df[c].var() for c in numeric_cols]
descriptive.round(2)


## 2. Distribution Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Sales"], kde=True, bins=12)
plt.title("Sales Distribution with Density Curve")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

normality_results = {}
for col in numeric_cols:
    statistic, p_value = stats.shapiro(df[col])
    normality_results[col] = {"Shapiro statistic": statistic, "p-value": p_value}

pd.DataFrame(normality_results).T.round(4)


## 3. Correlation Analysis

In [ ]:
correlation = df[numeric_cols].corr()
display(correlation.round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="Blues")
plt.title("Business Metrics Correlation Heatmap")
plt.tight_layout()
plt.show()


## 4. Hypothesis Testing

In [ ]:
# Test 1: one-sample t-test against a $45,000 benchmark
benchmark = 45000
t1, p1 = stats.ttest_1samp(df["Sales"], benchmark)

# Test 2: North vs South independent t-test
north = df.loc[df["Region"] == "North", "Sales"]
south = df.loc[df["Region"] == "South", "Sales"]
t2, p2 = stats.ttest_ind(north, south, equal_var=False)

# Test 3: one-way ANOVA across regions
groups = [df.loc[df["Region"] == r, "Sales"] for r in ["North", "South", "West"]]
f3, p3 = stats.f_oneway(*groups)

hypothesis_results = pd.DataFrame([
    ["One-sample t-test", t1, p1, "Significant" if p1 < 0.05 else "Not significant"],
    ["North vs South t-test", t2, p2, "Significant" if p2 < 0.05 else "Not significant"],
    ["Regional ANOVA", f3, p3, "Significant" if p3 < 0.05 else "Not significant"],
], columns=["Test", "Statistic", "p-value", "Decision"])

hypothesis_results


## 5. Confidence Interval

In [ ]:
mean_sales = df["Sales"].mean()
sem_sales = stats.sem(df["Sales"])
ci_low, ci_high = stats.t.interval(0.95, len(df)-1, loc=mean_sales, scale=sem_sales)
margin_of_error = mean_sales - ci_low

print(f"Mean Sales: ${mean_sales:,.2f}")
print(f"95% CI: ${ci_low:,.2f} to ${ci_high:,.2f}")
print(f"Margin of Error: ±${margin_of_error:,.2f}")


## 6. Linear Regression

In [ ]:
X = df[["Marketing_Spend"]]
y = df["Sales"]

model = LinearRegression()
model.fit(X, y)

predictions = model.predict(X)
r2 = r2_score(y, predictions)
rmse = mean_squared_error(y, predictions) ** 0.5

print(f"Intercept: {model.intercept_:,.2f}")
print(f"Marketing coefficient: {model.coef_[0]:.4f}")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:,.2f}")

plt.figure(figsize=(8, 5))
sns.regplot(x="Marketing_Spend", y="Sales", data=df, line_kws={"linewidth": 2})
plt.title("Marketing Spend vs Sales — Linear Regression")
plt.tight_layout()
plt.show()


## 7. Business Insights & Recommendations

1. Use the correlation result to understand how strongly marketing spend is associated with sales.
2. Use hypothesis-test p-values rather than averages alone when deciding whether observed differences are statistically meaningful.
3. Regional differences identified by the t-test/ANOVA can guide region-specific sales strategies.
4. The 95% confidence interval provides a reasonable range for the population mean daily sales.
5. R-squared indicates how much variation in sales is explained by marketing spend in the simple regression model.

### Conclusion
The analysis demonstrates the complete introductory statistics workflow required for the Week 7 internship task: descriptive statistics, distribution analysis, correlation, three hypothesis tests, confidence intervals, and regression.
